# 🇻🇳 Vietnamese OCR & LLM Correction Pipeline (`vieocr.ipynb`)
Notebook tương tác thử nghiệm và kiểm tra chất lượng OCR cho các tác phẩm tiếng Việt (scan mộc hoặc chữ Hán - Nôm đối chiếu).
Được thiết kế tối ưu cho **Sentence Alignment** (Đối chiếu câu Hán - Việt) với độ chính xác cao, tường minh, mạch lạc.

In [ ]:
# ---------------------------------------------------------
# 1. CẤU HÌNH & IMPORT THƯ VIỆN
# ---------------------------------------------------------
import os, sys, time, warnings, logging
import fitz, cv2
import numpy as np

PDF_REL_PATH = "data/raw/vie/an-nam-chi-nguyen/HVB_002_PDFScan_Viet_An Nam Chí Nguyên.pdf"
WORK_ID      = "HVB_002"
WORK_TITLE   = "An Nam Chí Nguyên"

START_PAGE   = 11   # Chỉ mục trang bắt đầu (0-indexed)
NUM_PAGES    = 1    # Số lượng trang cần xử lý (None = chạy đến hết file)
USE_LLM      = True # Bật/tắt LLM Corrector hậu xử lý và lọc rác

# Thêm đường dẫn src/run_ocr/vie vào sys.path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from ocr_utils import (
    find_file, init_paddleocr, init_vietocr,
    run_ocr_page, smart_sort_layout
)
from llm_corrector import correct_text_with_llm

print("✅ Đã import thành công các module ocr_utils và llm_corrector!")

In [ ]:
# ---------------------------------------------------------
# 2. LOAD & RENDER PDF THÀNH ẢNH (300 DPI)
# ---------------------------------------------------------
pdf_path = find_file(PDF_REL_PATH, WORK_ID)
if not pdf_path or not os.path.exists(pdf_path):
    raise FileNotFoundError(f"❌ Không tìm thấy file PDF tại: {PDF_REL_PATH}")

print(f"📂 Đang mở file PDF: {os.path.basename(pdf_path)}")
doc = fitz.open(pdf_path)
total_pages = len(doc)

end_page = min(START_PAGE + NUM_PAGES, total_pages) if NUM_PAGES else total_pages
print(f"📑 Sẽ chạy từ trang {START_PAGE + 1} đến trang {end_page} (Tổng: {end_page - START_PAGE} trang)...")

pages_images = []
for i in range(START_PAGE, end_page):
    page = doc[i]
    mat = fitz.Matrix(300 / 72, 300 / 72)  # Render chuẩn 300 DPI cho chữ Việt sắc nét
    pix = page.get_pixmap(matrix=mat)
    
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)
    pages_images.append(img)

doc.close()
print(f"✅ Đã render xong {len(pages_images)} trang ảnh!")

In [ ]:
# ---------------------------------------------------------
# 3. KHỞI TẠO ENGINE OCR (PADDLE + VIETOCR)
# ---------------------------------------------------------
print("⏳ Khởi tạo PaddleOCR (Detector DBNet)...", end=" ", flush=True)
paddle_engine = init_paddleocr(lang="vi")
print("OK ✅")

print("⏳ Khởi tạo VietOCR (vgg_transformer Recognizer)...", end=" ", flush=True)
vietocr_engine = init_vietocr()
print("OK ✅")

In [ ]:
# ---------------------------------------------------------
# 4. CHẠY PIPELINE OCR + HẬU XỬ LÝ LLM
# ---------------------------------------------------------
result_pages = []
raw_ocr_pages = []
all_ocr_lines_list = []
total_t0 = time.time()

print(f"\n🚀 Bắt đầu xử lý {len(pages_images)} trang...")
for idx, img in enumerate(pages_images):
    t0 = time.time()
    print(f"  → [Trang {idx + 1}/{len(pages_images)}] Nhận diện OCR...", end=" ", flush=True)
    
    # 1. Paddle detect box → crop → VietOCR đọc text
    ocr_lines = run_ocr_page(img, paddle_engine, vietocr_engine)
    all_ocr_lines_list.append(ocr_lines)
    
    # 2. Sắp xếp layout chuẩn từ trên xuống dưới, trái qua phải
    page_text = smart_sort_layout(ocr_lines)
    raw_ocr_pages.append(page_text)
    
    # 3. Hậu xử lý LLM (nối câu mạch lạc, sửa lỗi, xóa số trang phục vụ Sentence Alignment)
    if USE_LLM:
        try:
            final_text = correct_text_with_llm(page_text, work_title=WORK_TITLE, language="vie")
        except Exception as e:
            print(f"(LLM lỗi: {e}, giữ text Raw)", end=" ")
            final_text = page_text
    else:
        final_text = page_text
        
    result_pages.append(final_text)
    print(f"Xong ({time.time() - t0:.1f}s) - {len(ocr_lines)} dòng box.")

elapsed_total = time.time() - total_t0
print(f"\n🎉 HOÀN TẤT PIPELINE! Tổng thời gian: {elapsed_total:.2f} giây (~{elapsed_total/len(pages_images):.1f}s/trang).")

In [ ]:
# ---------------------------------------------------------
# 5. KIỂM TRA TRỰC QUAN BOUNDING BOX TRÊN ẢNH TRANG ĐẦU TIÊN
# ---------------------------------------------------------
import matplotlib.pyplot as plt

def draw_ocr_boxes(image, ocr_lines):
    """Vẽ bounding box màu xanh lá lên ảnh."""
    img_drawn = image.copy()
    for line in ocr_lines:
        box = np.array(line[0]).astype(np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_drawn, [box], isClosed=True, color=(0, 255, 0), thickness=2)
    return img_drawn

if pages_images and all_ocr_lines_list:
    img_with_boxes = draw_ocr_boxes(pages_images[0], all_ocr_lines_list[0])
    plt.figure(figsize=(12, 16))
    plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
    plt.title(f"Bounding Box Trang 1 ({len(all_ocr_lines_list[0])} vùng)")
    plt.axis("off")
    plt.show()

In [ ]:
# ---------------------------------------------------------
# 6. CHI TIẾT CÁC DÒNG BOX THÔ & ĐỘ TIN CẬY (SCORE) TRANG 1
# ---------------------------------------------------------
if all_ocr_lines_list:
    print("--- OCR Lines Trang 1 ---")
    for idx, line in enumerate(all_ocr_lines_list[0]):
        box, item = line
        text, score = item if isinstance(item, (tuple, list)) else (item, 1.0)
        print(f"{idx+1:2d}. [{float(score)*100:5.1f}%] : {text}")

In [ ]:
# ---------------------------------------------------------
# 7. SO SÁNH TRƯỚC VÀ SAU KHI SỬA LLM (TRANG 1)
# ---------------------------------------------------------
if raw_ocr_pages and result_pages:
    print("=" * 60)
    print("📄 TEXT RAW OCR TRƯỚC LLM:")
    print("=" * 60)
    print(raw_ocr_pages[0])
    print("\n" + "=" * 60)
    print("✨ TEXT HOÀN CHỈNH SAU LLM (SẴN SÀNG CHO SENTENCE ALIGNMENT):")
    print("=" * 60)
    print(result_pages[0])

In [ ]:
# # ---------------------------------------------------------
# # 8. XUẤT KẾT QUẢ HOÀN CHỈNH RA FILE .TXT
# # ---------------------------------------------------------
# output_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "..", "data", "ocr_output"))
# os.makedirs(output_dir, exist_ok=True)

# out_file = os.path.join(output_dir, f"{WORK_ID}_test_vie_raw.txt")
# final_full_text = "\n".join([page.strip() for page in result_pages if page.strip()])

# with open(out_file, "w", encoding="utf-8") as f:
#     f.write(final_full_text)

# print(f"💾 Đã lưu kết quả hoàn chỉnh vào:\n  👉 {out_file}\n")
# print("-" * 50)
# print("📋 PREVIEW 1000 KÝ TỰ ĐẦU TIÊN:")
# print("-" * 50)
# print(final_full_text[:1000])